<a href="https://colab.research.google.com/github/nisaa783/scikit-learn-cookbook_Anisa-Khasanah/blob/main/Chapter_12_Cross_Validation_and_Model_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Pendahuluan dan Pentingnya Validasi Silang (Cross-Validation)
Membagi data hanya sekali menjadi train set dan test set dapat membuat evaluasi model bergantung pada kebetulan pembagian data tersebut. Jika pembagian data kebetulan menempatkan sampel sulit atau tidak seimbang di salah satu bagian, nilai akurasi atau metrik evaluasi bisa menjadi kurang representatif terhadap performa model secara umum.

Cross-Validation mengatasi hal ini dengan membagi dataset ke dalam beberapa lipatan (folds), melatih model secara berulang pada sebagian lipatan, dan mengevaluasinya pada lipatan yang tersisa. Pada masalah klasifikasi dengan proporsi kelas yang tidak seimbang (imbalanced classes), Stratified K-Fold memastikan setiap lipatan mempertahankan proporsi label kelas yang sama seperti data aslinya.

## 2. Metrik Evaluasi Model dan Hyperparameter Tuning
Pemilihan metrik evaluasi harus disesuaikan dengan karakter masalah dan biaya kesalahan prediksi:
Klasifikasi: Selain Accuracy, metrik seperti Precision, Recall, F1-Score, dan ROC-AUC sangat penting ketika distribusi kelas tidak seimbang atau ketika false negative maupun false positive memiliki konsekuensi berbeda.
Regresi: Mean Squared Error (MSE), Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), dan koefisien determinasi ($R^2$) digunakan untuk mengukur selisih nilai prediksi terhadap nilai aktual.

Untuk mencari kombinasi hyperparameter terbaik tanpa mengintip test set berulang kali, GridSearchCV menguji kisi kombinasi parameter secara sistematis menggunakan validasi silang di dalam setiap kombinasi. Jika menggabungkan preprocessing dan model, memasukkan keduanya ke dalam Pipeline sebelum diteruskan ke GridSearchCV mencegah data leakage di setiap fold.

Konsep & Rumus Matematika:K-Fold Cross-Validation Score untuk $K$ lipatan dengan metrik skor $S_k$ pada lipatan ke-$k$:$$\text{CV}_{\text{score}} = \frac{1}{K} \sum_{k=1}^{K} S_k$$Precision, Recall, dan F1-Score (berdasarkan True Positive $TP$, False Positive $FP$, dan False Negative $FN$):$$\text{Precision} = \frac{TP}{TP + FP}, \quad \text{Recall} = \frac{TP}{TP + FN}, \quad F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$Root Mean Squared Error (RMSE) dan Coefficient of Determination ($R^2$) untuk $n$ sampel aktual $y_i$ dan prediksi $\hat{y}_i$:$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}, \quad R^2 = 1 - \frac{\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{n} (y_i - \bar{y})^2}$$

## 3. Contoh Program Penerapan 1: Stratified K-Fold Cross-Validation dan Evaluasi Metrik Klasifikasi
Contoh program pertama ini menerapkan StratifiedKFold dengan cross_validate untuk mengevaluasi beberapa metrik klasifikasi sekaligus pada dataset load_breast_cancer.



In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. Memuat dataset klasifikasi
data = load_breast_cancer()
X, y = data.data, data.target

# 2. Membangun pipeline preprocessing + model
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

# 3. Menyiapkan Stratified K-Fold (5 folds) dan menjalankan cross_validate
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring_metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]

cv_results = cross_validate(
    estimator=pipeline,
    X=X,
    y=y,
    cv=skf,
    scoring=scoring_metrics,
    return_train_score=True
)

# 4. Menampilkan rata-rata dan standar deviasi tiap metrik
print("Hasil Evaluasi 5-Fold Stratified Cross-Validation:")
for metric in scoring_metrics:
    test_scores = cv_results[f"test_{metric}"]
    train_scores = cv_results[f"train_{metric}"]
    print(
        f"{metric.upper():10s} | "
        f"Train: {train_scores.mean():.4f} (+/- {train_scores.std():.4f}) | "
        f"Test: {test_scores.mean():.4f} (+/- {test_scores.std():.4f})"
    )

Hasil Evaluasi 5-Fold Stratified Cross-Validation:
ACCURACY   | Train: 0.9881 (+/- 0.0011) | Test: 0.9737 (+/- 0.0166)
PRECISION  | Train: 0.9861 (+/- 0.0030) | Test: 0.9683 (+/- 0.0290)
RECALL     | Train: 0.9951 (+/- 0.0017) | Test: 0.9916 (+/- 0.0112)
F1         | Train: 0.9906 (+/- 0.0008) | Test: 0.9794 (+/- 0.0127)
ROC_AUC    | Train: 0.9977 (+/- 0.0010) | Test: 0.9953 (+/- 0.0053)


## 4. Contoh Program Penerapan 2: Tuning Hyperparameter dengan GridSearchCV dan Evaluasi Final pada Test Set

Contoh program kedua ini memisahkan data latih dan data uji terlebih dahulu, menjalankan GridSearchCV pada Pipeline menggunakan validasi silang pada data latih, lalu mengevaluasi estimator terbaik pada data uji yang terpisah.



In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# 1. Memuat dataset dan membagi train set serta test set
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Menyusun Pipeline dan grid hyperparameter
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(probability=True, random_state=42))
])

param_grid = {
    "svc__C": [0.1, 1.0, 10.0],
    "svc__gamma": ["scale", 0.01, 0.1],
    "svc__kernel": ["rbf", "linear"]
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 3. Pencarian hyperparameter terbaik berbasis skor F1
grid_search = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=cv_strategy,
    scoring="f1",
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print(f"Kombinasi Parameter Terbaik: {grid_search.best_params_}")
print(f"Skor F1 Rata-Rata Terbaik (CV): {grid_search.best_score_:.4f}")

# 4. Evaluasi model terbaik pada data uji (test set)
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print(f"ROC-AUC pada Data Uji: {roc_auc_score(y_test, y_prob):.4f}\n")
print("Laporan Klasifikasi pada Data Uji:")
print(classification_report(y_test, y_pred, target_names=data.target_names))

Kombinasi Parameter Terbaik: {'svc__C': 0.1, 'svc__gamma': 'scale', 'svc__kernel': 'linear'}
Skor F1 Rata-Rata Terbaik (CV): 0.9810
ROC-AUC pada Data Uji: 0.9937

Laporan Klasifikasi pada Data Uji:
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

